In [1]:
import os

In [2]:
%pwd

'c:\\Users\\yonas\\Desktop\\Mlops_Data_Science\\wine-quality-elastic-net\\notebooks'

In [3]:
os.chdir('../')
%pwd

'c:\\Users\\yonas\\Desktop\\Mlops_Data_Science\\wine-quality-elastic-net'

In [ ]:
from dataclasses import dataclass
from pathlib import Path

@dataclass
class DataIngestionConfig:
    root_dir :Path
    source_URL:str
    local_data_file:Path
    unzip_dir:Path

In [5]:
%pwd

'c:\\Users\\yonas\\Desktop\\Mlops_Data_Science\\wine-quality-elastic-net'

In [11]:
! pip install  joblib

  Using cached joblib-1.5.3-py3-none-any.whl.metadata (5.5 kB)
Using cached joblib-1.5.3-py3-none-any.whl (309 kB)


In [15]:
! pip install ensure

  Using cached ensure-1.0.4-py3-none-any.whl.metadata (10 kB)
Using cached ensure-1.0.4-py3-none-any.whl (15 kB)


In [17]:
! pip install python-box

  Using cached python_box-7.4.1-cp312-cp312-win_amd64.whl.metadata (8.3 kB)
Using cached python_box-7.4.1-cp312-cp312-win_amd64.whl (1.3 MB)


In [19]:
from src.constants.path import PARAMS_FILE_PATH,SCHEMA_FILE_PATH,CONFIG_FILE_PATH
from src.utils.common import read_yaml , create_directories

In [ ]:
class ConfigurationManager:
    def __init__(self , config_filepath =CONFIG_FILE_PATH, 
                 params_filepath=PARAMS_FILE_PATH , schema_filepath =SCHEMA_FILE_PATH):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)

        create_directories([self.config.artifacts_root])

    def get_data_ingestion_config(self) -> DataIngestionConfig:
        config = self.config.data_ingestion
        create_directories([config.root_dir])

        data_ingestion_config= DataIngestionConfig(
             root_dir = config.root_dir, 
             source_URL= config.source_URL,
             local_data_file = config.local_data_file,
             unzip_dir = config.unzip_dir
        )
        return data_ingestion_config
   
          

In [ ]:
import os
import urllib.request as request
from src.logger.logger import  logger
import zipfile

In [ ]:
## component-Data Ingestion

class DataIngestion:
    def __init__(self,config:DataIngestionConfig):
        self.config=config
    
    # Downloading the zip file
    def download_file(self):
        if not os.path.exists(self.config.local_data_file):
            filename, headers = request.urlretrieve(
                url = self.config.source_URL,
                filename = self.config.local_data_file
            )
            logger.info(f"{filename} download! with following info: \n{headers}")
        else:
            logger.info(f"File already exists")

    def extract_zip_file(self):
        """
        zip_file_path: str
        Extracts the zip file into the data directory
        Function returns None
        """
        unzip_path = self.config.unzip_dir
        os.makedirs(unzip_path, exist_ok=True)
        with zipfile.ZipFile(self.config.local_data_file, 'r') as zip_ref:
            zip_ref.extractall(unzip_path)


In [ ]:
try:
    config=ConfigurationManager()
    data_ingestion_config=config.get_data_ingestion_config()
    data_ingestion=DataIngestion(config=data_ingestion_config)
    data_ingestion.download_file()
    data_ingestion.extract_zip_file()
except Exception as e:
    raise e